# Ćwiczenie 2 – Podstawowe operacje arytmetyczne w analizie danych
Średnia, mediana, odchylenie standardowe, wariancja, korelacja i kowariancja policzone w dwóch zbiorach:
- **Część A** – ceny złota XAU/USD z Kaggle,
- **Część B** – częstość palenia tytoniu GBD 2019, dane z lab1.

# Część A – ceny złota XAU/USD

Dane: https://www.kaggle.com/datasets/novandraanugrah/xauusd-gold-price-historical-data-2004-2024

Plik `XAU_15m_data.csv` świece 15-minutowe, kolumny `Open`, `High`, `Low`, `Close`, `Volume`.

## Krok 1. Import bibliotek

In [1]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

## Krok 2. Wczytanie danych
Funkcja `dataset_load()` pobiera plik z Kaggle i zwraca ramkę danych pandas. Plik używa separatora `;`.

In [2]:
plik = "XAU_15m_data.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "novandraanugrah/xauusd-gold-price-historical-data-2004-2024",
    plik,
    pandas_kwargs={"delimiter": ";"},
)
df.head()

,Date,Open,High,Low,Close,Volume
0,2004.06.11 07:15,384.0,384.3,383.8,384.3,12
1,2004.06.11 07:30,383.8,384.3,383.6,383.8,12
2,2004.06.11 07:45,383.3,383.8,383.3,383.8,20
3,2004.06.11 08:00,383.8,384.1,383.6,383.6,8
4,2004.06.11 08:15,383.6,384.3,383.5,383.5,20


## Krok 3. Podstawowe informacje o zbiorze

In [3]:
display(df.shape)
display(df.isna().sum())
df.describe()

(494235, 6)

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

,Open,High,Low,Close,Volume
count,494235.000000,494235.000000,494235.000000,494235.000000,494235.000000
mean,1425.545439,1426.506119,1424.559039,1425.548296,988.000212
std,671.753822,672.309267,671.167825,671.764196,1758.706488
min,381.300000,381.600000,381.100000,381.300000,1.000000
25%,1086.765000,1087.550000,1085.950000,1086.760000,196.000000
50%,1305.940000,1306.630000,1305.230000,1305.940000,629.000000
75%,1757.320000,1758.510000,1756.110000,1757.350000,1195.000000
max,5585.790000,5597.600000,5577.410000,5585.900000,129960.000000


## Krok 4. Średnia arytmetyczna
Miara tendencji centralnej – suma wszystkich wartości podzielona przez ich liczbę:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

gdzie $x_1, \dots, x_n$ to wartości w zbiorze, $n$ to liczba obserwacji. Analizujemy kolumnę `High`.

In [4]:
print("Maksimum:", np.max(df["High"]))
print("Minimum: ", np.min(df["High"]))
print("Średnia: ", np.mean(df["High"]))

Maksimum: 5597.6
Minimum:  381.6
Średnia:  1426.5061187896447


## Krok 5. Mediana
Wartość środkowa w uporządkowanym zbiorze. Dla nieparzystego $n$ jest to element $x_{\frac{n+1}{2}}$, dla parzystego – średnia dwóch środkowych wartości:

$$\text{Mediana} = \begin{cases} x_{\frac{n+1}{2}} & n \text{ nieparzyste} \\ \frac{x_{n/2} + x_{n/2+1}}{2} & n \text{ parzyste} \end{cases}$$

Mediana jest odporna na wartości odstające.

In [5]:
n = len(df["High"])
print("Liczba obserwacji:", n)
print("Mediana numpy:  ", np.median(df["High"]))

posortowane = np.sort(df["High"].to_numpy())
mediana = posortowane[n // 2] if n % 2 else (posortowane[n // 2 - 1] + posortowane[n // 2]) / 2
print("Mediana ze wzoru:", mediana)

Liczba obserwacji: 494235
Mediana numpy:   1306.63
Mediana ze wzoru: 1306.63


## Krok 6. Odchylenie standardowe
Miara rozproszenia wartości względem średniej:

$$\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

W analizie finansowej służy do oceny zmienności i ryzyka.

In [6]:
print("Odchylenie standardowe numpy:   ", np.std(df["High"]))
print("Odchylenie standardowe ze wzoru:", np.sqrt(np.mean((df["High"] - df["High"].mean()) ** 2)))

Odchylenie standardowe numpy:    672.3085871081163
Odchylenie standardowe ze wzoru: 672.3085871081163


## Krok 7. Wariancja
Kwadrat odchylenia standardowego:

$$\text{Var}(X) = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

In [7]:
wariancja = np.var(df["High"])
print("Wariancja numpy:   ", wariancja)
print("Wariancja ze wzoru:", np.mean((df["High"] - df["High"].mean()) ** 2))
print("Odchylenie^2:        ", np.std(df["High"]) ** 2)

Wariancja numpy:    451998.8362993116
Wariancja ze wzoru: 451998.8362993116
Odchylenie^2:         451998.8362993116


## Krok 8. Korelacja
Korelacja Pearsona mierzy liniową zależność między dwiema zmiennymi:

$$\rho_{X,Y} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}$$

Badamy zależność między ceną najwyższą `High` i najniższą `Low`.

In [8]:
np.corrcoef(df["High"], df["Low"])[0, 1]

np.float64(0.9999952138297385)

## Krok 9. Kowariancja
Miara współzależności dwóch zmiennych:

$$\text{Cov}(X, Y) = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

Uwaga: `np.cov()` domyślnie dzieli przez $n-1$, a nie przez $n$ jak we wzorze. Wynik jest macierzą kowariancji: na przekątnej wariancje, poza nią kowariancja.

In [9]:
kowariancja = np.cov(df["High"], df["Low"])
kowariancja

array([[451999.75084351, 451230.18899553],
       [451230.18899553, 450466.24938756]])

## Krok 10. Korelacja jako iloraz kowariancji i odchyleń standardowych
Sprawdzamy wzór $\rho = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}$ na macierzy kowariancji. Ponieważ kowariancja i odchylenia pochodzą z tego samego estymatora, wynik jest identyczny z `np.corrcoef()`.

In [10]:
korelacja = kowariancja[0, 1] / (np.sqrt(kowariancja[0, 0]) * np.sqrt(kowariancja[1, 1]))
print("Ze wzoru:     ", korelacja)
print("np.corrcoef:  ", np.corrcoef(df["High"], df["Low"])[0, 1])
print("Zgodność:     ", np.isclose(korelacja, np.corrcoef(df["High"], df["Low"])[0, 1]))

Ze wzoru:      0.9999952138297385
np.corrcoef:   0.9999952138297385
Zgodność:      True


## Krok 11. Przykład z instrukcji – dane małe
Przykład z instrukcji: `data1 = [1..5]`, `data2 = [5..1]` – zależność liniowa ujemna, korelacja powinna wynosić dokładnie -1.

In [11]:
data1 = [1, 2, 3, 4, 5]
data2 = [5, 4, 3, 2, 1]
print("Średnia:     ", np.mean(data1))
print("Mediana:     ", np.median(data1))
print("Odch. std.:  ", np.std(data1))
print("Wariancja:   ", np.var(data1))
print("Korelacja:   ", np.corrcoef(data1, data2)[0, 1])
print("Kowariancja n-1:", np.cov(data1, data2)[0, 1])
print("Kowariancja n:  ", np.cov(data1, data2, bias=True)[0, 1])

Średnia:      3.0
Mediana:      3.0
Odch. std.:   1.4142135623730951
Wariancja:    2.0
Korelacja:    -0.9999999999999999
Kowariancja n-1: -2.5
Kowariancja n:   -2.0


## Krok 12. Przykład praktyczny – sprzedaż i ceny w dwóch sklepach
Zgodnie z instrukcją: dla danych o sprzedaży i cenach w dwóch sklepach liczymy średnią, medianę, odchylenie standardowe oraz korelację między ceną a sprzedażą. Dane są przykładowe; w sklepie B pojawia się wartość odstająca.

In [12]:
sklepy = pd.DataFrame({
    "cena_A":      [10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5, 15.0, 15.5],
    "sprzedaz_A":  [200, 195, 185, 180, 170, 165, 150, 148, 140, 130, 125, 120],
    "cena_B":      [9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5],
    "sprzedaz_B":  [220, 210, 205, 190, 185, 175, 170, 160, 150, 145, 135, 600],
})

wyniki = pd.DataFrame({
    "średnia":  [np.mean(sklepy[c]) for c in ["sprzedaz_A", "sprzedaz_B"]],
    "mediana":  [np.median(sklepy[c]) for c in ["sprzedaz_A", "sprzedaz_B"]],
    "odch_std": [np.std(sklepy[c]) for c in ["sprzedaz_A", "sprzedaz_B"]],
}, index=["Sklep A", "Sklep B"])
display(wyniki)

print("Korelacja cena-sprzedaż, sklep A:", np.corrcoef(sklepy["cena_A"], sklepy["sprzedaz_A"])[0, 1])
print("Korelacja cena-sprzedaż, sklep B:", np.corrcoef(sklepy["cena_B"], sklepy["sprzedaz_B"])[0, 1])
print("Korelacja sprzedaży A i B:       ", np.corrcoef(sklepy["sprzedaz_A"], sklepy["sprzedaz_B"])[0, 1])

,średnia,mediana,odch_std
Sklep A,159.000000,157.5,26.350206
Sklep B,212.083333,180.0,119.712474


Korelacja cena-sprzedaż, sklep A: -0.9967500038037731
Korelacja cena-sprzedaż, sklep B: 0.28281655167904185
Korelacja sprzedaży A i B:        -0.24700568659035085


## Wnioski do części A
- Dla ceny `High` złota średnia i mediana różnią się wyraźnie, a odchylenie standardowe jest duże – cena złota rosła przez 20 lat, więc rozkład jest silnie asymetryczny.
- Wariancja jest kwadratem odchylenia standardowego.
- Korelacja `High` i `Low` jest bliska 1 – w obrębie jednej świecy 15-minutowej ceny różnią się niewiele względem poziomu ceny.
- `np.cov()` dzieli przez $n-1$, a `np.var()` i `np.std()` domyślnie przez $n$; korelacja liczona z kowariancji i odchyleń z tego samego estymatora zgadza się z `np.corrcoef()`.
- W sklepie B wartość odstająca zawyża średnią do 212 przy medianie 180 oraz odchylenie standardowe do 120 przy 26 w sklepie A. Mediana jest odporniejsza na outliery.
- Korelacja cena-sprzedaż to około -1 w sklepie A, a w sklepie B przez tę wartość odstającą tylko 0,28. Korelację Pearsona też trzeba interpretować ostrożnie.

# Część B – częstość palenia tytoniu, dane z lab1

Zbiór **GBD 2019 – Smoking Tobacco Use Prevalence 1990-2019**. Kolumna `val` to standaryzowana wiekowo częstość palenia jako ułamek 0-1 dla kraju/regionu, płci i roku. Plik CSV jest w folderze `lab1`.

## Krok 1. Wczytanie danych
Plik szukamy obok notatnika lub w folderze `../lab1`.

In [13]:
from pathlib import Path

nazwa = "IHME_GBD_2019_SMOKING_TOB_1990_2019_PREV_Y2021M05D27.CSV"
plik = next(p for p in (Path(nazwa), Path("..") / "lab1" / nazwa) if p.exists())

df = pd.read_csv(plik).rename(columns={
    "location_name": "location", "sex_name": "sex", "year_id": "year",
})[["location", "sex", "year", "val", "upper", "lower"]]
display(df.shape)
display(df.isna().sum())
df.head()

(20970, 6)

location    0
sex         0
year        0
val         0
upper       0
lower       0
dtype: int64

,location,sex,year,val,upper,lower
0,Global,Male,1990,0.450607,0.453930,0.446915
1,Global,Female,1990,0.106411,0.108623,0.104415
2,Global,Both,1990,0.278179,0.280235,0.276236
3,Global,Male,1991,0.447212,0.450214,0.443670
4,Global,Female,1991,0.105164,0.107276,0.103293


## Krok 2. Średnia, minimum i maksimum
$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$
Liczymy dla całej kolumny `val` oraz osobno dla płci, pomijając rekordy `Both`.

In [14]:
print("Maksimum:", np.max(df["val"]))
print("Minimum: ", np.min(df["val"]))
print("Średnia: ", np.mean(df["val"]))
print()
plcie = df[df["sex"] != "Both"]
for p, g in plcie.groupby("sex"):
    print(f"Średnia {p}:", np.mean(g["val"]))

Maksimum: 0.656046804
Minimum:  0.006671118
Średnia:  0.2205751410672866

Średnia Female: 0.11919753006895564
Średnia Male: 0.3224456251955651


## Krok 3. Mediana
Wartość środkowa uporządkowanych danych; porównujemy wynik `np.median()` z obliczeniem ze wzoru i ze średnią.

In [15]:
x = np.sort(df["val"].to_numpy())
n = len(x)
mediana = x[n // 2] if n % 2 else (x[n // 2 - 1] + x[n // 2]) / 2
print("Liczba obserwacji:", n)
print("Mediana numpy: ", np.median(df["val"]))
print("Mediana wzór:  ", mediana)
print("Średnia:         ", np.mean(df["val"]))

Liczba obserwacji: 20970
Mediana numpy:  0.214082475
Mediana wzór:   0.214082475
Średnia:          0.2205751410672866


## Krok 4. Odchylenie standardowe i wariancja
$$\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2} \qquad \text{Var}(X) = \sigma^2$$

In [16]:
sr = df["val"].mean()
print("Odchylenie standardowe numpy:", np.std(df["val"]))
print("Odchylenie standardowe wzór: ", np.sqrt(np.mean((df["val"] - sr) ** 2)))
print("Wariancja numpy:             ", np.var(df["val"]))
print("Odchylenie^2:                  ", np.std(df["val"]) ** 2)

Odchylenie standardowe numpy: 0.13852964040595173
Odchylenie standardowe wzór:  0.13852964040595173
Wariancja numpy:              0.019190461271002292
Odchylenie^2:                   0.019190461271002292


## Krok 5. Porównanie miar dla mężczyzn i kobiet
Wszystkie miary w jednej tabeli.

In [17]:
tabela = plcie.groupby("sex")["val"].agg(
    srednia=np.mean, mediana=np.median, odch_std=np.std, wariancja=np.var
)
tabela

,srednia,mediana,odch_std,wariancja
sex,,,,
Female,0.119198,0.071666,0.103097,0.010629
Male,0.322446,0.321532,0.129380,0.016739


## Krok 6. Korelacja
$$\rho_{X,Y} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}$$
Najpierw `val` z granicami przedziału niepewności `lower` i `upper`, potem częstość palenia mężczyzn i kobiet w tym samym kraju i roku.

In [18]:
print("val vs lower:", np.corrcoef(df["val"], df["lower"])[0, 1])
print("val vs upper:", np.corrcoef(df["val"], df["upper"])[0, 1])

pivot = plcie.pivot_table(index=["location", "year"], columns="sex", values="val")
print("Mężczyźni vs kobiety:", np.corrcoef(pivot["Male"], pivot["Female"])[0, 1])

val vs lower: 0.9986216186697856
val vs upper: 0.9985559263489542
Mężczyźni vs kobiety: 0.4231894318322434


## Krok 7. Kowariancja
$$\text{Cov}(X, Y) = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$
`np.cov()` dzieli przez $n-1$ i zwraca macierz kowariancji.

In [19]:
kowariancja = np.cov(pivot["Male"], pivot["Female"])
display(kowariancja)
print("Cov n-1:", kowariancja[0, 1])
print("Cov n:  ", np.cov(pivot["Male"], pivot["Female"], bias=True)[0, 1])

array([[0.01687647, 0.00568168],
       [0.00568168, 0.01068075]])

Cov n-1: 0.005681675716659971
Cov n:   0.005680855850034191


## Krok 8. Korelacja jako iloraz kowariancji i odchyleń standardowych

In [20]:
korelacja = kowariancja[0, 1] / (np.sqrt(kowariancja[0, 0]) * np.sqrt(kowariancja[1, 1]))
print("Ze wzoru:   ", korelacja)
print("np.corrcoef:", np.corrcoef(pivot["Male"], pivot["Female"])[0, 1])
print("Zgodność:   ", np.isclose(korelacja, np.corrcoef(pivot["Male"], pivot["Female"])[0, 1]))

Ze wzoru:    0.42318943183224333
np.corrcoef: 0.4231894318322434
Zgodność:    True


## Krok 9. Trend w czasie – Polska
Korelacja roku z częstością palenia: wartość ujemna oznacza spadek w czasie.

In [21]:
pl = plcie[plcie["location"] == "Poland"]
for p, g in pl.groupby("sex"):
    print(f"{p}: korelacja rok-val = {np.corrcoef(g['year'], g['val'])[0, 1]:.4f}, "
          f"kowariancja = {np.cov(g['year'], g['val'])[0, 1]:.5f}")

Female: korelacja rok-val = -0.9191, kowariancja = -0.12187
Male: korelacja rok-val = -0.9592, kowariancja = -0.59372


## Krok 10. Wartości odstające – kraje z najwyższą częstością palenia
Sprawdzamy, jak skrajne rekordy wpływają na średnią i medianę.

In [22]:
rok = plcie[(plcie["year"] == 2019) & (plcie["sex"] == "Male")]["val"]
print("2019, mężczyźni – średnia:", np.mean(rok), " mediana:", np.median(rok), " odch. std.:", np.std(rok))
print("Maksimum:", rok.max(), "-", plcie.loc[rok.idxmax(), "location"])
bez = rok[rok < rok.quantile(0.95)]
print("Bez górnych 5%  – średnia:", np.mean(bez), " mediana:", np.median(bez))

2019, mężczyźni – średnia: 0.2928170515021459  mediana: 0.274798432  odch. std.: 0.12448175125904722
Maksimum: 0.646348913 - Timor-Leste
Bez górnych 5%  – średnia: 0.27865184157466066  mediana: 0.264071533


## Wnioski do części B
- Średnia częstość palenia w zbiorze to ok. 22%, mediana 21,4%, odchylenie standardowe 0,139 – rozkład jest prawie symetryczny, bo średnia i mediana są blisko.
- Mężczyźni palą średnio ponad 2,5 raza częściej niż kobiety, 32,2% wobec 11,9%. U kobiet mediana to 7,2% przy średniej 11,9%, bo kilka krajów z wysokim odsetkiem zawyża średnią.
- Korelacja `val` z `lower` i `upper` jest ≈ 0,999 – przedział niepewności leży tuż przy oszacowaniu, więc te kolumny nie wnoszą nowej informacji.
- Częstość palenia mężczyzn i kobiet w tym samym kraju i roku jest umiarkowanie skorelowana, ρ ≈ 0,42.
- W Polsce korelacja roku z częstością palenia jest silnie ujemna, −0,96 u mężczyzn i −0,92 u kobiet. Palenie spada w latach 1990-2019.
- W 2019 r. górne 5% krajów, z maksimum w Timor-Leste 64,6%, podnosi średnią dla mężczyzn z 27,9% do 29,3%, a medianę tylko z 26,4% do 27,5%.
- Zbiór zawiera też rekordy zbiorcze, np. `Global` i regiony, więc część obserwacji nakłada się na kraje.
- `np.cov()` dzieli przez n−1, a wzór z instrukcji przez n; przy 10 tys. obserwacji różnica jest pomijalna, 0,005682 wobec 0,005681.